# Workshop 2.2: Returns and Volatility

Welcome to Workshop 2.2! In Workshop 2.1, we learned how to download and cache historical stock prices. However, raw dollar prices alone cannot tell us whether a strategy is performing well.

### Why Quants Analyze Returns Instead of Raw Prices

A five-dollar price increase means something radically different for a ten-dollar penny stock than for a thousand-dollar tech giant. On a ten-dollar stock, that five-dollar rise represents a massive fifty percent surge, while on a thousand-dollar stock it is barely noticeable.

To evaluate assets on an equal footing, quantitative finance converts raw price paths into **percentage returns**. Alongside returns, we evaluate **volatility**, which measures the statistical dispersion of those price swings and serves as our foundational gauge of investment risk.

In this workshop, we will calculate daily returns, compare simple versus logarithmic math, and learn the square-root-of-time rule for annualizing risk.

> **Key Takeaway**: Converting raw dollar prices into percentage returns normalizes assets across different price scales, while volatility quantifies investment risk.

## Topic 1: Calculating Daily Returns

The most common way to measure price movement is the **simple percentage return**:
$$\text{Return}_t = \frac{\text{Price}_t - \text{Price}_{t-1}}{\text{Price}_{t-1}} = \frac{\text{Price}_t}{\text{Price}_{t-1}} - 1$$

In pandas, we calculate this across entire columns in a single vectorized step using **`.pct_change()`**.

Let's load our cached Apple dataset and compute daily percentage returns. Let's see:

In [1]:
import pandas as pd
import numpy as np

# Load AAPL data from Workshop 2.1:
try:
    aapl = pd.read_parquet("aapl_2023.parquet")
except Exception:
    import yfinance as yf
    aapl = yf.download("AAPL", start="2023-01-01", end="2024-01-01", progress=False)
    aapl.index = aapl.index.tz_localize(None)

# Calculate percentage return:
aapl["Return"] = aapl["Close"].pct_change()

print(aapl[["Close", "Return"]].head())

                 Close    Return
Date                            
2023-01-03  125.070000       NaN
2023-01-04  126.360001  0.010314
2023-01-05  125.019997 -0.010605
2023-01-06  129.619995  0.036794
2023-01-09  130.149994  0.004089
2023-01-10  130.740005  0.004533
2023-01-11  133.490005  0.021034
2023-01-12  133.410004 -0.000600
2023-01-13  134.759995  0.010119
2023-01-17  135.940002  0.008756


### Why the First Return Row Shows NaN

Notice that the first row displays `NaN`. To calculate a return, pandas requires both today's price and yesterday's price. Because our dataset begins on January 3, 2023, no prior observation exists to compute a percentage change for that opening row.

> **Key Takeaway**: `.pct_change()` calculates percentage returns row by row, naturally producing `NaN` on the initial row where no prior baseline exists.

## Topic 2: Understanding Returns: Simple vs Logarithmic

In financial mathematics, returns are modeled in two ways:
- **Simple Returns**: $R_t = (P_t / P_{t-1}) - 1$. Simple returns reflect real-world cash accounting and portfolio rebalancing.
- **Logarithmic Returns**: $r_t = \ln(P_t / P_{t-1})$. Log returns offer statistical advantages because multi-period returns sum together additively.

Log returns possess symmetrical mathematical properties: a positive ten percent move followed by a negative ten percent log drop brings your cumulative calculation back to zero. For typical daily market moves of one or two percent, simple and log returns are virtually identical.

Let's compare simple and log returns side by side. Let's see:

In [2]:
# Compute simple returns:
aapl["Simple_Return"] = aapl["Close"].pct_change()

# Compute log returns:
aapl["Log_Return"] = np.log(aapl["Close"] / aapl["Close"].shift(1))

print(aapl[["Close", "Simple_Return", "Log_Return"]].head())

                 Close  Simple_Return  Log_Return
Date                                             
2023-01-03  125.070000            NaN         NaN
2023-01-04  126.360001       0.010314    0.010261
2023-01-05  125.019997      -0.010605   -0.010662
2023-01-06  129.619995       0.036794    0.036133
2023-01-09  130.149994       0.004089    0.004080


> **Key Takeaway**: Simple returns represent intuitive portfolio percentage changes, while log returns offer additive convenience in statistical modeling.

---

## Topic 3: Calculating Volatility as Return Dispersion

In quantitative finance, **volatility** is defined as the statistical standard deviation of percentage returns over a chosen observation window.

Think of volatility like wave height in an ocean: calm seas show minimal height fluctuations, while choppy waters exhibit violent swings. In financial markets, low volatility indicates steady, predictable price changes, whereas high volatility reflects wide price turbulence.

In equity trading, a 20-day rolling window corresponds to approximately one calendar trading month. We calculate rolling monthly volatility using `.rolling(20).std()`.

Let's calculate our rolling 20-day volatility. Let's check:

In [3]:
# Calculate 20-day rolling standard deviation of daily returns:
aapl["Volatility_20d"] = aapl["Return"].rolling(20).std()

print("Rolling 20-Day Volatility (last 5 rows):")
print(aapl[["Close", "Return", "Volatility_20d"]].tail())

Rolling 20-Day Volatility (last 5 rows):
                 Close    Return  Volatility_20d
Date                                            
2023-12-22  193.600006 -0.005547        0.007629
2023-12-26  193.050003 -0.002841        0.007636
2023-12-27  193.149994  0.000518        0.007559
2023-12-28  193.580002  0.002226        0.007537
2023-12-29  192.529999 -0.005424        0.007615


> **Key Takeaway**: Volatility measures the standard deviation of percentage returns, capturing the magnitude of market price swings.

---

## Topic 4: Annualizing Returns and Volatility

Institutional investors rarely evaluate performance using daily fractions like 0.08 percent. Instead, strategies are benchmarked on an **annualized** basis.

Because stock exchanges close on weekends and official holidays, a standard trading year contains **252 trading days** rather than 365 calendar days.

### Annualizing Compounded Returns
To annualize a daily mean return, we compound it over 252 trading sessions:
$$\text{Annualized Return} = (1 + \text{Daily Mean Return})^{252} - 1$$

### Annualizing Volatility via the Square Root of Time Rule
Because financial variance expands linearly across time, standard deviation scales with the square root of time $\sqrt{t}$:
$$\text{Annualized Volatility} = \text{Daily Volatility} \times \sqrt{252}$$

This square root scaling often surprises newcomers, but it reflects how random price fluctuations disperse over extended horizons.

Let's calculate both annualized metrics for Apple in 2023. Let's see:

In [4]:
# We calculate average daily return and annualize it:
daily_mean_return = aapl["Return"].mean()
annualized_return = (1 + daily_mean_return) ** 252 - 1

# We calculate daily volatility and annualize it:
daily_volatility = aapl["Return"].std()
annualized_volatility = daily_volatility * np.sqrt(252)

print("=== AAPL 2023 Annualized Performance ===")
print(f"Daily Mean Return:     {daily_mean_return:.6f} ({daily_mean_return * 100:.3f}% per day)")
print(f"Annualized Return:     {annualized_return:.4f} ({annualized_return * 100:.2f}% per year)")
print(f"Daily Volatility:      {daily_volatility:.6f} ({daily_volatility * 100:.3f}% per day)")
print(f"Annualized Volatility: {annualized_volatility:.6f} ({annualized_volatility * 100:.2f}% per year)")

=== AAPL 2023 Annualized Performance ===
Daily Mean Return:     0.001928 (0.193% per day)
Annualized Return:     0.6253 (62.53% per year)
Daily Volatility:      0.012547 (1.255% per day)
Annualized Volatility: 0.199175 (19.92% per year)


> **Key Takeaway**: We assume 252 trading days per year, compounding returns over 252 days and scaling volatility by $\sqrt{252}$.

---

## Topic 5: Visualizing Price and Return Distributions

Visualizing time-series data helps us spot structural shifts and outlier shocks that summary tables might obscure.

Using `matplotlib`, we frequently generate two primary visual diagnostics:
- **Price History Plot**: Displays directional trends across the full calendar horizon.
- **Return Distribution Histogram**: Reveals the dispersion and tail risk of daily price movements.

Let's plot Apple's closing price trajectory. Let's see:

In [5]:
import matplotlib.pyplot as plt

print("Displaying AAPL Closing Price chart for 2023:")
aapl["Close"].plot(figsize=(10, 5), title="AAPL Closing Price 2023")
plt.show()

Displaying AAPL Closing Price chart for 2023:


Now let's inspect the distribution of daily returns.

Notice how the histogram centers around zero with bell-curve symmetry, punctuated by occasional fat tails on high-volatility news sessions. Let's check:

In [6]:
print("Displaying Distribution of AAPL Daily Returns (Histogram):")
aapl["Return"].hist(bins=50, figsize=(10, 5), title="Distribution of AAPL Daily Returns")
plt.show()

Displaying Distribution of AAPL Daily Returns (Histogram):


> **Key Takeaway**: Plotting price trends and return distributions exposes market dynamics and tail risk that summary statistics cannot capture alone.

---

## Practice Time

Now it is your turn to calculate volatility metrics and compare investment risk profiles. Measuring risk accurately is central to quantitative engineering, so work through these calculations deliberately.

---

### Challenge 1: Rolling Volatility Windows

- Using the `aapl` DataFrame, compute a 30-day rolling volatility on the `"Return"` column.
- Store the output in a new column named `"Volatility_30d"`.
- Display the last 5 rows of `["Close", "Return", "Volatility_30d"]`.

In [ ]:
# Challenge 1: Write your code below this line


# Expected: Displays the last 5 rows of AAPL with 30-day rolling volatility.


### Challenge 2: Annualized Return Comparison

- Load the Microsoft dataset (`"msft_2023.parquet"`) from Workshop 2.1.
- Calculate daily percentage returns for MSFT using `.pct_change()`.
- Compute and display the annualized return for MSFT using the 252-day compounding formula.

In [ ]:
# Challenge 2: Write your code below this line


# Expected Output:
# MSFT Annualized Return: ~0.65 (around 65%)


### Challenge 3: Annualized Volatility and Risk Profiling

- Calculate the annualized volatility for MSFT using `msft["Return"].std() * np.sqrt(252)`.
- Display the annualized volatility for MSFT.
- Compare it against Apple's volatility figure, printing a conclusion on which asset proved more volatile in 2023.

In [ ]:
# Challenge 3: Write your code below this line


# Expected: Prints MSFT annualized volatility (~21.9%) and comparison conclusion.


---

## Solutions Section

Terrific work completing these financial calculations! Measuring returns and risk accurately allows quants to compare assets and evaluate investment strategies objectively.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
aapl["Volatility_30d"] = aapl["Return"].rolling(30).std()
print("AAPL 30-Day Volatility (last 5 rows):")
print(aapl[["Close", "Return", "Volatility_30d"]].tail())
```

#### Solution for Challenge 2:
```python
try:
    msft = pd.read_parquet("msft_2023.parquet")
except Exception:
    import yfinance as yf
    msft = yf.download("MSFT", start="2023-01-01", end="2024-01-01", progress=False)
    msft.index = msft.index.tz_localize(None)

msft["Return"] = msft["Close"].pct_change()
msft_daily_mean = msft["Return"].mean()
msft_annualized_return = (1 + msft_daily_mean) ** 252 - 1
print(f"MSFT Annualized Return: {msft_annualized_return:.4f} ({msft_annualized_return * 100:.2f}%)")
```

#### Solution for Challenge 3:
```python
msft_daily_vol = msft["Return"].std()
msft_annualized_vol = msft_daily_vol * np.sqrt(252)
print(f"MSFT Annualized Volatility: {msft_annualized_vol:.4f} ({msft_annualized_vol * 100:.2f}%)")
print(f"AAPL Annualized Volatility: {annualized_volatility:.4f} ({annualized_volatility * 100:.2f}%)")
if msft_annualized_vol > annualized_volatility:
    print("MSFT was more volatile than AAPL in 2023.")
else:
    print("AAPL was more volatile than MSFT in 2023.")
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [7]:
# Solution for Challenge 1:
aapl["Volatility_30d"] = aapl["Return"].rolling(30).std()
print("AAPL 30-Day Volatility (last 5 rows):")
print(aapl[["Close", "Return", "Volatility_30d"]].tail())

AAPL 30-Day Volatility (last 5 rows):
                 Close    Return  Volatility_30d
Date                                            
2023-12-22  193.600006 -0.005547        0.008455
2023-12-26  193.050003 -0.002841        0.008461
2023-12-27  193.149994  0.000518        0.008459
2023-12-28  193.580002  0.002226        0.008398
2023-12-29  192.529999 -0.005424        0.008447


In [8]:
# Solution for Challenge 2:
try:
    msft = pd.read_parquet("msft_2023.parquet")
except Exception:
    import yfinance as yf
    msft = yf.download("MSFT", start="2023-01-01", end="2024-01-01", progress=False)
    msft.index = msft.index.tz_localize(None)

msft["Return"] = msft["Close"].pct_change()
msft_daily_mean = msft["Return"].mean()
msft_annualized_return = (1 + msft_daily_mean) ** 252 - 1
print(f"MSFT Annualized Return: {msft_annualized_return:.4f} ({msft_annualized_return * 100:.2f}%)")

MSFT Annualized Return: 0.6514 (65.14%)


In [9]:
# Solution for Challenge 3:
msft_daily_vol = msft["Return"].std()
msft_annualized_vol = msft_daily_vol * np.sqrt(252)
print(f"MSFT Annualized Volatility: {msft_annualized_vol:.4f} ({msft_annualized_vol * 100:.2f}%)")
print(f"AAPL Annualized Volatility: {annualized_volatility:.4f} ({annualized_volatility * 100:.2f}%)")
if msft_annualized_vol > annualized_volatility:
    print("MSFT was more volatile than AAPL in 2023.")
else:
    print("AAPL was more volatile than MSFT in 2023.")

MSFT Annualized Volatility: 0.2195 (21.95%)
AAPL Annualized Volatility: 0.1992 (19.92%)
MSFT was more volatile than AAPL in 2023.
